# Gold Layer Data Monitoring
Validate row counts, schema consistency, join integrity, and dropped rows between Silver → Gold transformations.

In [ ]:
USE DATABASE LEAGUE_RECORDS;

USE SCHEMA GOLD;

## 1. Object Inventory
Take stocks of all items under the Gold Layer.
1. `CHAMPION_INTERVALS`: Champion aggregated performance over time.
2. `CHAMPION_OVERVIEW`: Champion aggregated info e.g. win-rates, ban-rates, etc.
3. `ITEM_STATS_AND_RECOMMENDATIONS`: Champion and item usage rate, recommendations, etc.
4. `MATCH_TEAM_STATS_SUMMARY`: Match-grain pivot views.
5. `PLAYER_STATS_SUMMARY`: Match-player end game snapshot.

In [ ]:
%%sql -r gold_all_dynamic_tables
SHOW DYNAMIC TABLES IN SCHEMA GOLD;

## 2. Importing from Silver 
Validate that gold grain matches expectations:
- `PLAYER_STATS_SUMMARY` should have 1 row per (MATCH_ID, PARTICIPANT_POS_ID) = same distinct combos as PLAYERS_SUMMARY_SILVER
- `MATCH_TEAM_STATS_SUMMARY` should have 1 row per MATCH_ID = same count as MATCHES_SUMMARY_SILVER
- `CHAMPION_OVERVIEW` row count ≈ distinct champions in CHAMPIONS_REF_SILVER (minus ID=0)

Identify any dropped records between `SILVER` and `GOLD`.

In [ ]:
WITH CHECKS AS (
    SELECT
        'PLAYER_STATS_SUMMARY' AS CHECK_NAME,
        (SELECT COUNT(*) FROM GOLD.PLAYER_STATS_SUMMARY) AS GOLD_ROWS,
        (SELECT COUNT(DISTINCT MATCH_ID || '|' || PARTICIPANT_POS_ID) FROM SILVER.PLAYERS_SUMMARY_SILVER) AS EXPECTED_ROWS
    UNION ALL
    SELECT
        'MATCH_TEAM_STATS_SUMMARY',
        (SELECT COUNT(*) FROM GOLD.MATCH_TEAM_STATS_SUMMARY),
        (SELECT COUNT(DISTINCT MATCH_ID) FROM SILVER.TEAM_INTERVAL_SILVER)
    UNION ALL
    SELECT
        'CHAMPION_OVERVIEW',
        (SELECT COUNT(*) FROM GOLD.CHAMPION_OVERVIEW),
        (SELECT COUNT(*) FROM SILVER.CHAMPIONS_REF_SILVER WHERE CHAMPION_ID != 0)
)

SELECT
    CHECK_NAME,
    GOLD_ROWS,
    EXPECTED_ROWS,
    GOLD_ROWS - EXPECTED_ROWS AS ROW_DIFF,
    CASE
        WHEN GOLD_ROWS = EXPECTED_ROWS THEN 'PASS'
        WHEN GOLD_ROWS < EXPECTED_ROWS THEN 'ROWS DROPPED'
        ELSE '⚠ MORE ROWS THAN EXPECTED'
    END AS STATUS
FROM CHECKS;

In [ ]:
WITH SILVER_KEYS AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM SILVER.PLAYERS_SUMMARY_SILVER
),

GOLD_KEYS AS (
    SELECT DISTINCT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.PLAYER_STATS_SUMMARY
),

MISSING AS (
    SELECT S.MATCH_ID, S.PARTICIPANT_POS_ID
    FROM SILVER_KEYS AS S
    LEFT JOIN GOLD_KEYS AS G
        ON S.MATCH_ID = G.MATCH_ID 
        AND S.PARTICIPANT_POS_ID = G.PARTICIPANT_POS_ID
    WHERE G.MATCH_ID IS NULL
)

SELECT
    COUNT(*) AS ROWS_DROPPED,
    (SELECT COUNT(*) FROM SILVER_KEYS) AS TOTAL_SILVER_ROWS,
    ROUND(
        COUNT(*) / (SELECT COUNT(*) FROM SILVER_KEYS) * 100
    , 2) AS PCT_DROPPED
FROM MISSING;

In [ ]:
WITH SILVER_MATCHES AS (
    SELECT DISTINCT MATCH_ID 
    FROM SILVER.MATCHES_SUMMARY_SILVER
),

GOLD_MATCHES AS (
    SELECT DISTINCT MATCH_ID 
    FROM GOLD.MATCH_TEAM_STATS_SUMMARY
),

MISSING AS (
    SELECT S.MATCH_ID
    FROM SILVER_MATCHES AS S
    LEFT JOIN GOLD_MATCHES AS G 
        ON S.MATCH_ID = G.MATCH_ID
    WHERE G.MATCH_ID IS NULL
)

SELECT
    COUNT(*) AS MATCHES_DROPPED,
    (SELECT COUNT(*) FROM SILVER_MATCHES) AS TOTAL_SILVER_MATCHES,
    ROUND(
        COUNT(*) / (SELECT COUNT(*) FROM SILVER_MATCHES) * 100
    , 2) AS PCT_DROPPED
FROM MISSING;

In [ ]:
SELECT * 
FROM BRONZE._UNLOGGED_MATCHES 
ORDER BY UNLOGGED_AT_LOAD_DATE 
LIMIT 10
;

## 3. Data Quality Monitoring
* Freshness: Table all refreshed within setup target lags.
* Duplicate Key Detection: No duplication on primary key. Each table's grain should be unique.
* Abnormal Volume Detection: Each table contains a volume of records within acceptable limits.
* Per-table accuracy, statistics, uniqueness: Check each table for logical values.

In [ ]:
WITH REFRESH_STATUS AS (
    SELECT 
        'PLAYER_STATS_SUMMARY' AS GOLD_TABLE, 
        '1 DAY' AS TARGET_LAG, 
        36 AS MAX_HRS,
        COALESCE(MAX_BY(STATE, REFRESH_START_TIME), 'NO_HISTORY') AS LAST_STATE,
        DATEDIFF('HOUR', MAX(REFRESH_END_TIME), CURRENT_TIMESTAMP()) AS HRS_SINCE
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME => 'PLAYER_STATS_SUMMARY'))
        UNION ALL
    SELECT 'MATCH_TEAM_STATS_SUMMARY', '1 DAY', 36,
           COALESCE(MAX_BY(STATE, REFRESH_START_TIME), 'NO_HISTORY'),
           DATEDIFF('HOUR', MAX(REFRESH_END_TIME), CURRENT_TIMESTAMP())
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME => 'MATCH_TEAM_STATS_SUMMARY'))
        UNION ALL
    SELECT 'CHAMPION_OVERVIEW', '1 DAY', 36,
           COALESCE(MAX_BY(STATE, REFRESH_START_TIME), 'NO_HISTORY'),
           DATEDIFF('HOUR', MAX(REFRESH_END_TIME), CURRENT_TIMESTAMP())
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME => 'CHAMPION_OVERVIEW'))
        UNION ALL
    SELECT 'CHAMPION_INTERVALS', '1 DAY', 36,
           COALESCE(MAX_BY(STATE, REFRESH_START_TIME), 'NO_HISTORY'),
           DATEDIFF('HOUR', MAX(REFRESH_END_TIME), CURRENT_TIMESTAMP())
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME => 'CHAMPION_INTERVALS'))
        UNION ALL
    SELECT 'ITEM_STATS_AND_RECOMMENDATIONS', '7 DAYS', 216,
           COALESCE(MAX_BY(STATE, REFRESH_START_TIME), 'NO_HISTORY'),
           DATEDIFF('HOUR', MAX(REFRESH_END_TIME), CURRENT_TIMESTAMP())
    FROM TABLE(INFORMATION_SCHEMA.DYNAMIC_TABLE_REFRESH_HISTORY(NAME => 'ITEM_STATS_AND_RECOMMENDATIONS'))
)

SELECT
    GOLD_TABLE,
    TARGET_LAG,
    LAST_STATE,
    HRS_SINCE,
    IFF(LAST_STATE = 'SUCCEEDED' AND HRS_SINCE <= MAX_HRS, 'PASS', 'WARN') AS STATUS
FROM REFRESH_STATUS
ORDER BY GOLD_TABLE;

In [ ]:
%%sql -r duplicate_key_reports
SELECT 
    'PLAYER_STATS_SUMMARY' AS GOLD_TABLE, 
    'MATCH_ID + PARTICIPANT_POS_ID' AS GRAIN,
    COUNT(*) - COUNT(DISTINCT MATCH_ID || '|' || PARTICIPANT_POS_ID) AS DUPLICATE_ROWS
FROM GOLD.PLAYER_STATS_SUMMARY
    UNION ALL
SELECT 
    'MATCH_TEAM_STATS_SUMMARY',
    'MATCH_ID',
    COUNT(*) - COUNT(DISTINCT MATCH_ID)
FROM GOLD.MATCH_TEAM_STATS_SUMMARY
    UNION ALL
SELECT 
    'CHAMPION_OVERVIEW',
    'CHAMPION_ID',
    COUNT(*) - COUNT(DISTINCT CHAMPION_ID)
FROM GOLD.CHAMPION_OVERVIEW
    UNION ALL
SELECT 
    'CHAMPION_INTERVALS',
    'CHAMPION + MINUTE',
    COUNT(*) - COUNT(DISTINCT CHAMPION || '|' || MINUTE)
FROM GOLD.CHAMPION_INTERVALS
    UNION ALL
SELECT 
    'ITEM_STATS_AND_RECOMMENDATIONS',
    'ITEM + CHAMPIONS',
    COUNT(*) - COUNT(DISTINCT CHAMPION || '|' || ITEM)
FROM GOLD.ITEM_STATS_AND_RECOMMENDATIONS
;

In [ ]:
%%sql -r abnormal_volume_reports
WITH GOLD_VOLUME AS (
    SELECT 
        'PLAYER_STATS_SUMMARY' AS GOLD_TABLE, 
        COUNT(*) AS GOLD_ROWS 
    FROM GOLD.PLAYER_STATS_SUMMARY
        UNION ALL 
    SELECT 'MATCH_TEAM_STATS_SUMMARY', COUNT(*) FROM GOLD.MATCH_TEAM_STATS_SUMMARY
        UNION ALL 
    SELECT 'CHAMPION_OVERVIEW', COUNT(*) FROM GOLD.CHAMPION_OVERVIEW
        UNION ALL
    SELECT 'CHAMPION_INTERVALS', COUNT(*) FROM GOLD.CHAMPION_INTERVALS
        UNION ALL 
    SELECT 'ITEM_STATS_AND_RECOMMENDATIONS', COUNT(*) FROM GOLD.ITEM_STATS_AND_RECOMMENDATIONS
),

UPSTREAM AS (
    SELECT
        (SELECT COUNT(*) FROM SILVER.PLAYERS_SUMMARY_SILVER) AS N_PLAYERS,
        (SELECT COUNT(*) FROM SILVER.MATCHES_SUMMARY_SILVER) AS N_MATCHES,
        (SELECT COUNT(DISTINCT CASE WHEN CHAMPION = 'Fiddle Sticks' THEN 'Fiddlesticks' ELSE CHAMPION END)
         FROM SILVER.PLAYERS_SUMMARY_SILVER) AS N_CHAMPIONS,
        (SELECT COUNT(DISTINCT MINUTE) FROM SILVER.PLAYER_INTERVAL_SILVER) AS N_MINUTES,
        (SELECT COUNT(*) FROM SILVER.ITEMS_REF_SILVER) AS N_ITEMS
),

BOUNDS AS (
    SELECT 
        'PLAYER_STATS_SUMMARY' AS GOLD_TABLE, 
        'VS PLAYERS_SUMMARY_SILVER (<=5% DROP)' AS BASIS,
        CEIL(0.95 * N_PLAYERS) AS LOWER_BOUND, 
        N_PLAYERS AS UPPER_BOUND 
    FROM UPSTREAM
        UNION ALL
    SELECT 
        'MATCH_TEAM_STATS_SUMMARY', 
        'VS MATCHES_SUMMARY_SILVER (<=5% DROP)',
        CEIL(0.95 * N_MATCHES), N_MATCHES 
    FROM UPSTREAM
        UNION ALL
    SELECT 
        'CHAMPION_OVERVIEW', 
        'VS DISTINCT CHAMPIONS IN PLAYERS (<=5% DROP)',
        CEIL(0.95 * N_CHAMPIONS), N_CHAMPIONS 
    FROM UPSTREAM
        UNION ALL
    SELECT 
        'CHAMPION_INTERVALS',  
        'BETWEEN #CHAMPIONS AND #CHAMPIONS * #MINUTES',
        N_CHAMPIONS, 
        N_CHAMPIONS * N_MINUTES 
    FROM UPSTREAM
        UNION ALL
    SELECT 
        'ITEM_STATS_AND_RECOMMENDATIONS', 
        'BETWEEN #CHAMPIONS AND #CHAMPIONS * #ITEMS',
        N_CHAMPIONS, 
        N_CHAMPIONS * N_ITEMS 
    FROM UPSTREAM
)

SELECT
    B.GOLD_TABLE,
    G.GOLD_ROWS,
    B.LOWER_BOUND,
    B.UPPER_BOUND,
    B.BASIS,
    CASE 
        WHEN G.GOLD_ROWS BETWEEN B.LOWER_BOUND AND B.UPPER_BOUND THEN 'PASS'
        ELSE 'FAIL'
    END AS STATUS
FROM BOUNDS AS B
JOIN GOLD_VOLUME AS G USING (GOLD_TABLE)
ORDER BY B.GOLD_TABLE;

Per table accuracy, statistics, null checks.


In [ ]:
WITH M AS (
    SELECT
        COUNT(*) AS N,
        SUM(IFF(
            LEVEL NOT BETWEEN 1 AND 20
            OR LEAST(KILLS, DEATHS, ASSISTS, CS, TOTAL_GOLD) < 0
            OR UNLOGGED_DURATION < 0
            OR TEAM NOT IN ('Blue', 'Red')
        , 1, 0)) AS WEIRD_VALUES,
        SUM(IFF(
            MATCH_ID IS NULL 
            OR PARTICIPANT_POS_ID IS NULL
            OR CHAMPION IS NULL 
            OR WIN IS NULL
        , 1, 0)) AS NULL_KEYS
    FROM GOLD.PLAYER_STATS_SUMMARY
)

SELECT 
    'LOGICAL VALUES' AS CHECK_NAME,
    IFF(WEIRD_VALUES = 0, 'PASS', 'FAIL') AS STATUS,
    'TOTAL_WEIRD_VALUES=' || WEIRD_VALUES || ' OF ' || N AS DETAIL 
FROM M
    UNION ALL
SELECT 
    'NO NULLS IN KEYS', 
    IFF(NULL_KEYS = 0, 'PASS', 'FAIL'), 
    'NULL_ROWS=' || NULL_KEYS || ' OF ' || N 
FROM M

ORDER BY 1;

In [ ]:
WITH M AS (
    SELECT
        COUNT(*) AS N,
        SUM(IFF(
            AVG_LEVEL NOT BETWEEN 1 AND 20
            OR LEAST(AVG_CUMU_KILLS, AVG_CUMU_DEATHS, AVG_CUMU_ASSISTS, AVG_CUMU_CS, AVG_CUMU_TOTAL_GOLD) < 0
            OR ROWS_SAMPLED < 1
            OR MINUTE < 0
        , 1, 0)) AS WEIRD_VALUES,
        SUM(IFF(
            CHAMPION IS NULL OR 
            MINUTE IS NULL
        , 1, 0)) AS NULL_KEYS
    FROM GOLD.CHAMPION_INTERVALS
)

SELECT 
    'LOGICAL VALUES' AS CHECK_NAME,
    IFF(WEIRD_VALUES = 0, 'PASS', 'FAIL') AS STATUS,
    'VIOLATIONS=' || WEIRD_VALUES || ' OF ' || N AS DETAIL
FROM M
    UNION ALL
SELECT 
    'NO NULLS IN KEYS',
   IFF(NULL_KEYS = 0, 'PASS', 'FAIL'), 
   'NULL_ROWS=' || NULL_KEYS || ' OF ' || N 
FROM M

ORDER BY 1;

In [ ]:
WITH M AS (
    SELECT
        COUNT(*) AS N,
        SUM(IFF(
            (GLOBAL_WIN_RATE IS NOT NULL AND GLOBAL_WIN_RATE NOT BETWEEN 0 AND 1)
            OR GLOBAL_BAN_RATE NOT BETWEEN 0 AND 1
            OR PRIMARY_LANE_SHARE NOT BETWEEN 0 AND 1
            OR GLOBAL_PICK_RATE < 0
            OR GLOBAL_GAMES_PLAYED < 0
            OR MOST_PICKED_LANE IS NULL
        , 1, 0)) AS WEIRD_VALUES,
        SUM(IFF(
            CHAMPION_ID IS NULL 
            OR CHAMPION_NAME IS NULL
        , 1, 0)) AS NULL_KEYS,
        SUM(IFF(
            GLOBAL_WIN_RATE IS NOT NULL 
            AND GLOBAL_GAMES_PLAYED >= 30
            AND GLOBAL_WIN_RATE NOT BETWEEN 0.40 AND 0.60
        , 1, 0)) AS OUT_OF_BAND
    FROM GOLD.CHAMPION_OVERVIEW
),

X_VALID AS (
    SELECT 
    FROM GOLD.CHAMPION_OVERVIEW
),



SELECT 
    'LOGICAL VALUES' AS CHECK_NAME,
    IFF(WEIRD_VALUES = 0, 'PASS', 'FAIL') AS STATUS,
    'VIOLATIONS=' || WEIRD_VALUES || ' OF ' || N AS DETAIL 
FROM M
    UNION ALL
SELECT 
    'NO NULLS IN CHAMPION_ID / CHAMPION_NAME',
    IFF(NULL_KEYS = 0, 'PASS', 'FAIL'), 
    'NULL_ROWS=' || NULL_KEYS || ' OF ' || N 
FROM M
    UNION ALL
SELECT 
    'WIN-RATE PLAUSIBILITY (GAMES>=30)',
    IFF(OUT_OF_BAND = 0, 'PASS', 'WARN'), 
    'OUT_OF_BAND_CHAMPIONS=' || OUT_OF_BAND 
FROM M
    
ORDER BY 1;

In [ ]:
%%sql -r audit_match_stats_reports
WITH M AS (
    SELECT
        COUNT(*) AS N,
        SUM(IFF(
            WINNING_TEAM NOT IN ('Blue', 'Red')
            OR GAME_DURATION < 0
            OR UNLOGGED_DURATION < 0
            OR GAME_DATE IS NULL
            OR GAME_DATE > CURRENT_DATE
            OR BLUE_KILLS IS NULL OR RED_KILLS IS NULL
            OR BLUE_TOWERS IS NULL OR RED_TOWERS IS NULL
            OR BLUE_DRAGONS IS NULL OR RED_DRAGONS IS NULL
            OR BLUE_VOID_GRUBS IS NULL OR RED_VOID_GRUBS IS NULL
            OR BLUE_HERALDS IS NULL OR RED_HERALDS IS NULL
            OR BLUE_BARONS IS NULL OR RED_BARONS IS NULL
            OR LEAST(BLUE_KILLS, RED_KILLS, BLUE_TOWERS, RED_TOWERS,
                     BLUE_DRAGONS, RED_DRAGONS, BLUE_VOID_GRUBS, RED_VOID_GRUBS,
                     BLUE_HERALDS, RED_HERALDS, BLUE_BARONS, RED_BARONS) < 0
        , 1, 0)) AS BAD_DOMAIN,
        SUM(IFF(
            BLUE_TOWERS > 11 OR RED_TOWERS > 11
            OR BLUE_BARONS > 6 OR RED_BARONS > 6
            OR BLUE_DRAGONS > 8 OR RED_DRAGONS > 8
            OR BLUE_HERALDS > 2 OR RED_HERALDS > 2
            OR BLUE_VOID_GRUBS > 6 OR RED_VOID_GRUBS > 6
        , 1, 0)) AS OBJ_ANOMALY,
        AVG(IFF(WINNING_TEAM = 'Blue', 1.0, 0.0)) AS BLUE_WIN
    FROM GOLD.MATCH_TEAM_STATS_SUMMARY
)
SELECT 'DOMAIN BOUNDS + DATE SANITY (NOT NULL, NOT FUTURE)' AS CHECK_NAME,
       'ACCURACY' AS DIMENSION, 'HARD' AS SEVERITY,
       IFF(BAD_DOMAIN = 0, 'PASS', 'FAIL') AS STATUS,
       'VIOLATIONS=' || BAD_DOMAIN || ' OF ' || N AS DETAIL FROM M
UNION ALL
SELECT 'OBJECTIVE CEILINGS (TOWERS<=11, BARONS<=6, DRAGONS<=8, ...)', 'CUSTOM', 'WARN',
       IFF(OBJ_ANOMALY = 0, 'PASS', 'WARN'), 'IMPOSSIBLE_OBJECTIVE_ROWS=' || OBJ_ANOMALY FROM M
UNION ALL
SELECT 'BLUE-SIDE WIN SHARE 0.45-0.58', 'STATISTICS', 'WARN',
       IFF(BLUE_WIN BETWEEN 0.45 AND 0.58, 'PASS', 'WARN'),
       'BLUE_WIN=' || TO_VARCHAR(ROUND(BLUE_WIN, 4)) FROM M
ORDER BY 1;

In [ ]:
%%sql -r audit_items_report
WITH M AS (
    SELECT
        COUNT(*) AS N,
        SUM(IFF(
            PLAYER_PURCHASE_RATE NOT BETWEEN 0 AND 1
            OR WIN_RATE NOT BETWEEN 0 AND 1
            OR AVG_KDA < 0
            OR (MOST_COMMON_FIRST_PURCHASE_MINUTE IS NOT NULL
                AND MOST_COMMON_FIRST_PURCHASE_MINUTE < 0)
            OR ITEM IS NULL 
            OR ITEM_CATEGORY IS NULL
        , 1, 0)) AS WEIRD_VALUES,
        SUM(IFF(
            CHAMPION IS NULL
            ITEM IS NULL
            OR ITEM_CATEGORY IS NULL
            

        , 1, 0)) AS NULL_VALUES,
        SUM(IFF(
            ITEM = TOP_ITEM_1 
            OR ITEM = TOP_ITEM_2 
            OR ITEM = TOP_ITEM_3
            OR PLAYER_PURCHASE_RATE > 0.99
        , 1, 0)) AS BROKEN_RECS
    FROM GOLD.ITEM_STATS_AND_RECOMMENDATIONS
)

SELECT 
    'DOMAIN BOUNDS (RATES 0-1, KDA>=0, ITEM NOT NULL)' AS CHECK_NAME,
    IFF(WEIRD_VALUES = 0, 'PASS', 'FAIL') AS STATUS,
    'VIOLATIONS=' || WEIRD_VALUES || ' OF ' || N AS DETAIL 
FROM M
    UNION ALL
SELECT 
    'RECOMMENDATION SANITY', 
    IFF(BROKEN_RECS = 0, 'PASS', 'WARN'), 
    'ANOMALOUS_ROWS=' || BROKEN_RECS 
FROM M

ORDER BY 1;

## 4. Sample Records: Gold vs Silver
Pull a few records and compare between silver and gold they carry the same informations.

In [ ]:
WITH SAMPLE_PLAYERS AS (
    SELECT MATCH_ID, PARTICIPANT_POS_ID
    FROM GOLD.PLAYER_STATS_SUMMARY
    ORDER BY RANDOM()
    LIMIT 30
),

SILVER_LAST_INTERVAL AS (
    SELECT *
    FROM SILVER.PLAYER_INTERVAL_SILVER
    QUALIFY ROW_NUMBER() OVER (
        PARTITION BY MATCH_ID, PARTICIPANT_POS_ID
        ORDER BY MINUTE DESC
    ) = 1
)

SELECT
    -- Primary key
    G.MATCH_ID AS SAMPLED_MATCH_ID,
    G.PARTICIPANT_POS_ID AS SAMPLED_PARTICIPANT_POS_ID,
    -- The sampled player's records in GOLD.PLAYER_STATS_SUMMARY
    G.KILLS AS GOLD_KILLS,
    G.DEATHS AS GOLD_DEATHS,
    G.ASSISTS AS GOLD_ASSISTS,
    G.TOTAL_GOLD AS GOLD_TOTAL_GOLD,
    -- The sampled player's records in SILVER.PLAYER_INTERVAL_SILVER
    PIV.KILLS AS SILVER_LAST_KILLS,
    PIV.DEATHS AS SILVER_LAST_DEATHS,
    PIV.ASSISTS AS SILVER_LAST_ASSISTS,
    PIV.TOTAL_GOLD AS SILVER_LAST_GOLD,
    PIV.MINUTE AS SILVER_LAST_MINUTE,
    -- Verify joined from Gold is same as silver
    CASE 
        WHEN G.KILLS = PIV.KILLS 
            AND G.DEATHS = PIV.DEATHS
            AND G.ASSISTS = PIV.ASSISTS 
            AND G.TOTAL_GOLD = PIV.TOTAL_GOLD
            THEN '✓ MATCH'
        ELSE '✗ MISMATCH' 
    END AS VERIFICATION
FROM GOLD.PLAYER_STATS_SUMMARY AS G
JOIN SAMPLE_PLAYERS AS SP
    ON G.MATCH_ID = SP.MATCH_ID 
    AND G.PARTICIPANT_POS_ID = SP.PARTICIPANT_POS_ID
JOIN SILVER_LAST_INTERVAL AS PIV
    ON PIV.MATCH_ID = SP.MATCH_ID
    AND PIV.PARTICIPANT_POS_ID = SP.PARTICIPANT_POS_ID;

# Summary Health

In [ ]:
# Aggregate health results into a summary score
import pandas as pd

checks = []

# Grain checks
grain_df = grain_checks.to_pandas()
for _, row in grain_df.iterrows():
    checks.append({
        'check': row['CHECK_NAME'],
        'passed': str(row['STATUS']).startswith('✓')
    })

# Duplicate checks
dup_df = duplicate_check.to_pandas()
for _, row in dup_df.iterrows():
    checks.append({
        'check': f"No duplicates in {row['GOLD_TABLE']}",
        'passed': int(row['DUPLICATE_ROWS']) == 0
    })

# NULL audit
null_df = null_audit.to_pandas()
for _, row in null_df.iterrows():
    checks.append({
        'check': f"No NULLs in {row['GOLD_TABLE']}.{row['COLUMN_NAME']}",
        'passed': int(row['NULL_COUNT']) == 0
    })

df = pd.DataFrame(checks)
passed = df['passed'].sum()
total = len(df)

print(f"{'='*50}")
print(f"  GOLD LAYER HEALTH SCORE: {passed}/{total} checks passed")
print(f"{'='*50}")
print()
for _, row in df.iterrows():
    icon = '✓' if row['passed'] else '✗'
    print(f"  {icon}  {row['check']}")

if passed < total:
    print(f"\n  ⚠ {total - passed} issue(s) require investigation.")
else:
    print(f"\n  All checks passed. Gold layer is consistent with silver.")